<a href="https://colab.research.google.com/github/ninobronw/BPALM-TOOL-IMPLEMENTATION/blob/main/Full_Code_NLP_RAG_Project_Notebook_JLA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Install llama-cpp-python (used to load the local GGUF model).
# IMPORTANT: we do NOT use --force-reinstall or CMAKE here.
# That destructive rebuild is exactly what corrupted numpy in the previous notebook.
# A plain install pulls a ready-made wheel and leaves numpy untouched.
!pip install -q llama-cpp-python


**What this cell does (easy English):** it installs `llama-cpp-python`, the tool that runs the local Mistral model. **Why no `--force-reinstall`:** that option rebuilds packages and broke `numpy` last time. **What to observe:** a short install log. Run it once, then move on.

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [2]:
# Core libraries. numpy is pinned to 2.3.3 so the whole stack stays consistent.
# We added the MODERN packages langchain-huggingface and langchain-chroma
# (they replace the deprecated embedding / vector-store imports).
!pip install -q huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 langchain-huggingface langchain-chroma chromadb==1.1.1 sentence-transformers==5.1.1 numpy==2.3.3


**What this cell does:** it installs all the other libraries and pins `numpy` to 2.3.3 so nothing conflicts. A yellow dependency warning (e.g. about `numba`) is **normal and safe to ignore**. **After this cell + the model libs: restart the runtime once, then run every cell top to bottom and DO NOT run the install cells again.**

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [3]:
# Libraries for processing data / text
import json, os
import tiktoken
import pandas as pd

# Loading, chunking, embedding, vector database  (MODERN, non-deprecated imports)
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings   # was: langchain_community...SentenceTransformerEmbeddings (deprecated)
from langchain_chroma import Chroma                       # was: langchain_community.vectorstores.Chroma (deprecated)

# Downloading and loading the LLM
from huggingface_hub import hf_hub_download
from llama_cpp import Llama


**What this cell does:** it imports the tools. The embedding and vector-store imports now use the **modern** packages (`langchain_huggingface`, `langchain_chroma`), which removes the deprecation warnings that confused the previous notebook.

## Question Answering using LLM

#### Downloading and Loading the model

In [4]:
# Download the Mistral-7B Instruct model in GGUF format (a small, quantized file that runs locally).
model_path = hf_hub_download(
    repo_id="TheBloke/Mistral-7B-Instruct-v0.1-GGUF",
    filename="mistral-7b-instruct-v0.1.Q4_K_M.gguf",
)

# Load the model with llama-cpp.
#   n_ctx=2048      -> how much text the model can read at once (context window)
#   n_gpu_layers=0  -> run on CPU (safe everywhere; no GPU build needed, so numpy stays intact)
llm = Llama(
    model_path=model_path,
    n_ctx=2048,
    n_gpu_layers=0,
    verbose=False,
)
print("LLM object initialized successfully!")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


LLM object initialized successfully!


**What this cell does:** it downloads the Mistral-7B model file and loads it. **What to observe:** `LLM object initialized successfully!`. The model now lives in the variable `llm`.

#### Response

In [5]:
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

**What this cell does:** it defines `response()`, a small helper that sends a plain text question to the model and returns the generated text. Nothing is printed yet — we only defined the function.

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [6]:
query_1 = "What is the protocol for managing sepsis in a critical care unit?"
print(response(query_1))




Sepsis is a life-threatening condition that requires prompt and aggressive management in a critical care unit. The protocol for managing sepsis in a critical care unit typically involves the following steps:

1. Early recognition: Sepsis should be suspected in any patient who is critically ill and has a suspected or confirmed infection.
2. Rapid diagnosis: Blood cultures should be obtained as soon as possible to identify the causative pathogen.
3. Appropriate antibiotic therapy: Antibiotics should be started as soon as the causative pathogen is identified.
4.


**This section (Question Answering using LLM):** we ask the model the 5 questions **without** any medical context, just to see what it knows on its own. The answers may be vague or partly wrong — that is expected and shows why RAG is useful.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [7]:
query_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(response(query_2))


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [8]:
query_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(response(query_3))





Sudden patchy hair loss, also known as alopecia areata, is a common hair loss condition that affects both men and women. It is characterized by the appearance of small, round or oval-shaped bald spots on the scalp. The exact cause of alopecia areata is not fully understood, but it is believed to be an autoimmune disorder that affects the hair follicles.

There are several effective treatments and solutions for addressing sudden patchy hair loss, including:

1. Topical corticosteroids: These are anti-inflammatory


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [9]:
query_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(response(query_4))





1. Rest and Recovery: The first step in treating a brain injury is to allow the brain to heal naturally. This may involve rest, sleep, and avoiding activities that could further damage the brain.

2. Medical Treatment: Depending on the severity of the injury, medical treatment may be necessary. This could include medication to manage pain or swelling, surgery to remove damaged tissue, or other treatments to help the brain heal.

3. Rehabilitation: Rehabilitation is an important part of the recovery process for a brain injury. This may involve physical therapy, occupational therapy,


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [10]:
query_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(response(query_5))






1. Assess the severity of the fracture: The first step is to assess the severity of the fracture. This can be done by examining the leg and feeling for any deformities or misalignments. If the fracture is severe, it may require immediate medical attention.

2. Immobilize the leg: Once the severity of the fracture has been assessed, the leg should be immobilized to prevent further damage. This can be done by using a cast, brace, or splint.

3. Pain management: Pain management is important to ensure


## Question Answering using LLM with Prompt Engineering

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [11]:
prompt = f"""[INST] You are a careful medical assistant.
Answer the question clearly, step by step, and stay factual.
Question: {query_1} [/INST]"""
print(response(prompt))


 Sepsis is a life-threatening condition that requires prompt and aggressive management in a critical care unit. The protocol for managing sepsis in a critical care unit typically involves the following steps:

1. Recognition and diagnosis: Sepsis should be suspected in any patient who is critically ill and has a suspected or confirmed infection. The diagnosis of sepsis is based on the presence of systemic inflammatory response syndrome (SIRS) and infection.
2. Resuscitation: The first step in managing sepsis is to resuscitate the patient with fluid and electrolyte


**This section (Prompt Engineering):** same 5 questions, but we wrap them with clearer instructions (`[INST] ... [/INST]`). Better instructions usually give cleaner answers, but the model still has **no access to the manual** yet.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [12]:
prompt = f"""[INST] You are a careful medical assistant.
Answer the question clearly, step by step, and stay factual.
Question: {query_2} [/INST]"""
print(response(prompt))


 Appendicitis is a medical condition that occurs when the appendix, a small organ attached to the large intestine, becomes inflamed or infected. The common symptoms of appendicitis include:
1. Abdominal pain: The pain usually starts in the lower right side of the abdomen and may spread to the upper right side.
2. Nausea and vomiting: These symptoms may occur with or without abdominal pain.
3. Loss of appetite: In some cases, the patient may lose their appetite due to the pain or nausea.
4. Fever: A low-


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [13]:
prompt = f"""[INST] You are a careful medical assistant.
Answer the question clearly, step by step, and stay factual.
Question: {query_3} [/INST]"""
print(response(prompt))


 Sudden patchy hair loss, commonly seen as localized bald spots on the scalp, can be caused by several factors. The effective treatments or solutions for addressing sudden patchy hair loss depend on the underlying cause. Here are some possible causes and treatments for sudden patchy hair loss:

1. Alopecia Areata: Alopecia areata is an autoimmune disorder that causes hair loss in patches. The treatment for alopecia areata includes topical corticosteroids, immunomodulators, and minoxidil.

2. Dermatitis: D


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [14]:
prompt = f"""[INST] You are a careful medical assistant.
Answer the question clearly, step by step, and stay factual.
Question: {query_4} [/INST]"""
print(response(prompt))


 Treatment for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, typically involves a multidisciplinary approach that includes medical, surgical, and rehabilitation interventions. Here are some of the recommended treatments:
1. Medical Interventions: Depending on the severity of the injury, the person may require immediate medical attention. This may include emergency surgery to remove any damaged tissue or to relieve pressure on the brain. In addition, the person may be given medications to manage pain, reduce inflammation, and prevent seizures.
2.


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [15]:
prompt = f"""[INST] You are a careful medical assistant.
Answer the question clearly, step by step, and stay factual.
Question: {query_5} [/INST]"""
print(response(prompt))


 If a person has fractured their leg during a hiking trip, the following precautions and treatment steps should be taken:
1. Stay Calm: The first step is to remain calm and assess the situation. Do not panic, and try to keep the person still to prevent further damage to the leg.
2. Call for Help: If possible, call for medical assistance or a rescue team to transport the person to a hospital or medical facility.
3. Immobilize the Leg: Use a splint or brace to immobilize the leg and prevent further movement. This will help to reduce pain and prevent further


## Data Preparation for RAG

### Loading the Data

In [16]:
from langchain_community.document_loaders import PyMuPDFLoader

# Make sure the file "medical_diagnosis_manual.pdf" is uploaded to this session first
# (left panel -> Files -> Upload), then load every page of the Merck Manual.
loader = PyMuPDFLoader("medical_diagnosis_manual.pdf")
data = loader.load()
print(f"Loaded {len(data)} pages")

Loaded 4114 pages


**What this cell does:** it reads the Merck Manual PDF and loads each page as a document. **What to observe:** about 4,000+ pages loaded.

### Data Overview

#### Checking the first 5 pages

In [17]:
# Look at the first 5 pages to check the text was read correctly.
for page in data[:5]:
    print(page.page_content[:500])
    print("-" * 80)


jlabena@gmail.com
7IJP65QWB1
This file is meant for personal use by jlabena@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
--------------------------------------------------------------------------------
jlabena@gmail.com
7IJP65QWB1
This file is meant for personal use by jlabena@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
--------------------------------------------------------------------------------
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .............................

**What this cell does:** it prints the start of the first 5 pages so we can confirm the text was extracted correctly.

#### Checking the number of pages

In [18]:
# How many pages did we load?
print(f"Number of pages: {len(data)}")


Number of pages: 4114


**What this cell does:** it prints the total number of pages — a quick sanity check.

### Data Chunking

In [19]:
# Split the long pages into small overlapping pieces ("chunks").
# Small pieces are easier to search and fit better into the model's context window.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,     # about 512 characters per chunk
    chunk_overlap=16,   # a small overlap so we do not cut ideas in half
)
chunks = text_splitter.split_documents(data)
print(f"Number of chunks: {len(chunks)}")


Number of chunks: 31030


**What this cell does:** it cuts the long pages into small overlapping **chunks**. Smaller pieces are easier to search and fit the model's context window. **What to observe:** several thousand chunks.

### Embedding

In [20]:
# Turn each text chunk into a list of numbers (a "vector") so the computer can compare meanings.
# gte-small produces vectors of size 384.
embedding_model = HuggingFaceEmbeddings(model_name="thenlper/gte-small")
print("Embedding dimension:", len(embedding_model.embed_query("test sentence")))


modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/66.7M [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384


**What this cell does:** it loads the embedding model that turns text into numeric vectors (size **384**). These vectors let us compare meaning, not just words.

### Vector Database

In [21]:
# Store all chunk vectors in a Chroma vector database so we can search them quickly.
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="merck_manual",
)
print("Vector database created.")


Vector database created.


**What this cell does:** it stores every chunk vector in a **Chroma** database so we can search them in milliseconds.

### Retriever

In [22]:
# A retriever fetches the 3 most similar chunks for any question (k=3).
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Retriever ready (k=3)")


Retriever ready (k=3)


**What this cell does:** it builds the **retriever**, which returns the 3 chunks most related to any question (`k=3`). This is the 'R' (retrieval) in RAG.

### System and User Prompt Template

In [23]:
# These two texts tell the model HOW to behave during RAG.
# The placeholders {context} and {question} are filled in later by the function.
qna_system_message = """You are an assistant to a medical professional.
Answer the user's question using ONLY the context provided.
The context comes after ###Context and the question after ###Question.
Do not talk about the context itself in your answer.
If the answer is not in the context, reply exactly: I don't know."""

qna_user_message_template = """###Context
{context}

###Question
{question}
"""


**What this cell does:** it writes the rules (`qna_system_message`) and the fill-in template (`qna_user_message_template`). The placeholders `{context}` and `{question}` are filled automatically later. They force the model to answer **only from the manual**.

### Response Function

In [24]:
def generate_rag_response(user_input, k=3, max_tokens=128, temperature=0, top_p=0.95, top_k=50):
    global qna_system_message, qna_user_message_template
    # Retrieve relevant document chunks (modern API: retriever.invoke instead of deprecated get_relevant_documents)
    relevant_document_chunks = retriever.invoke(user_input)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    # Fill the template with the context and the question
    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
        )
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response


**What this cell does:** it defines `generate_rag_response()`, the full RAG pipeline — *retrieve the best chunks, build the prompt, ask the model*. This is where retrieval and generation come together.

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [25]:
print(generate_rag_response(query_1))


Answer: The protocol for managing sepsis in a critical care unit includes monitoring systemic pressure, CVP, PAOP, or both; pulse oximetry; ABGs; blood glucose, lactate, and electrolyte levels; renal function, and possibly sublingual PCO2. Urine output should be measured, usually with an indwelling catheter. Fluid resuscitation with 0.9% saline should be given until CVP reaches 8 mm Hg (10 cm H2O) or PAOP. The patient should be evaluated for


**This section (Question Answering using RAG):** now the same 5 questions are answered **using the manual**. Compare these answers with the two earlier sections — they should be more precise and grounded in the Merck Manual.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [26]:
print(generate_rag_response(query_2))


###Answer
The common symptoms for appendicitis are abdominal pain, anorexia, and abdominal tenderness. It cannot be cured via medicine. If not treated, it can lead to necrosis, gangrene, and perforation. The surgical procedure to treat it is open or laparoscopic appendectomy.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [27]:
print(generate_rag_response(query_3))


Answer:
Sudden patchy hair loss, commonly seen as localized bald spots on the scalp, could be caused by alopecia areata, an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers. The Merck Manual of Diagnosis & Therapy recommends a suspension of 10% salicylic acid in mineral oil as a treatment for scalp plaques, which may be rubbed into the scalp at bedtime manually or with a toothbrush, covered with a shower cap, and washed out the next morning with a tar


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [28]:
print(generate_rag_response(query_4))


Answer:
Rehabilitation services should be planned early for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function. The specific rehabilitation therapy varies depending on the patient's abnormalities, which depend on the level and extent (partial or complete) of the injury. Complete transsection causes flaccid paralysis, while partial transsection causes cognitive and emotional impairments. Surgery may be needed in some cases. Cognitive therapy is often begun immediately after injury and continued for months or years.


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [29]:
print(generate_rag_response(query_5))


I don't know.


### Fine-tuning

In [30]:
# Fine-tuning is discussed in the markdown note above. No code is run in this cell:
# for this project we rely on RAG (retrieval) rather than fine-tuning the model's weights,
# because RAG is cheaper, faster to update, and keeps the answers tied to the manual.


**Fine-tuning :** fine-tuning means re-training the model's own weights on medical data. We do **not** do it here because it is expensive and slow to update. RAG gives us fresh, manual-based answers without changing the model, which is the right choice for this project.

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [ ]:
groundedness_rater_system_message = """You are a strict evaluator.
You receive a CONTEXT, a QUESTION and an ANSWER.
Rate GROUNDEDNESS: how much the answer is supported by the context.
Reply with an integer score from 1 (not grounded) to 5 (fully grounded) and one short reason."""


**This section (Output Evaluation):** we use the model as a **judge** (LLM-as-a-judge). Below we define two judges — one for **groundedness** (is the answer based on the context?) and one for **relevance** (does it answer the question?) — plus the template that feeds them the context, question and answer.

In [ ]:
relevance_rater_system_message = """You are a strict evaluator.
You receive a CONTEXT, a QUESTION and an ANSWER.
Rate RELEVANCE: how well the answer addresses the question.
Reply with an integer score from 1 (not relevant) to 5 (fully relevant) and one short reason."""


In [42]:
user_message_template = """###Context
{context}

###Question
{question}

###Answer
{answer}
"""


In [34]:
def generate_ground_relevance_response(user_input, k=3, max_tokens=128, temperature=0, top_p=0.95, top_k=50):
    global qna_system_message, qna_user_message_template
    # Retrieve relevant document chunks (modern API: retriever.invoke)
    relevant_document_chunks = retriever.invoke(user_input)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # First, generate the RAG answer that we want to evaluate
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""
    response = llm(prompt=prompt, max_tokens=max_tokens, temperature=temperature,
                   top_p=top_p, top_k=top_k, stop=['INST'])
    answer = response["choices"][0]["text"]

    # Ask the model (as a judge) to rate groundedness
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Ask the model (as a judge) to rate relevance
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(prompt=groundedness_prompt, max_tokens=max_tokens, temperature=temperature,
                     top_p=top_p, top_k=top_k, stop=['INST'])
    response_2 = llm(prompt=relevance_prompt, max_tokens=max_tokens, temperature=temperature,
                     top_p=top_p, top_k=top_k, stop=['INST'])

    return response_1['choices'][0]['text'], response_2['choices'][0]['text']


**What this cell does:** `generate_ground_relevance_response()` first produces a RAG answer, then asks the judge twice (groundedness, then relevance) and returns both scores.

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [35]:
groundedness, relevance = generate_ground_relevance_response(query_1)
print("GROUNDEDNESS:", groundedness)
print("RELEVANCE:", relevance)


GROUNDEDNESS:  4: The answer is fully grounded in the context. It accurately lists all the recommended monitoring and treatment protocols for managing sepsis in a critical care unit, including specific measurements and tests.
RELEVANCE:  RELEVANCE: 4

The answer provides a comprehensive overview of the protocol for managing sepsis in a critical care unit, including the specific monitoring and treatment measures. However, it does not provide a clear explanation of the prognosis or the specific causes of septic shock that were mentioned in the context.


**This section:** we run the evaluation on the 5 questions. **What to observe:** for each question, a groundedness score and a relevance score (1 to 5) with a short reason. High scores mean the system stayed faithful to the manual and answered the question.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [36]:
groundedness, relevance = generate_ground_relevance_response(query_2)
print("GROUNDEDNESS:", groundedness)
print("RELEVANCE:", relevance)


GROUNDEDNESS:  GROUNDEDNESS: 5

The answer is fully grounded in the context provided. It accurately identifies the common symptoms of appendicitis, states that it cannot be cured via medicine, and provides the correct surgical procedure to treat it. The answer also explains the etiology of appendicitis and the potential complications if it is left untreated, which further supports its accuracy.
RELEVANCE:  RELEVANCE: 5

The answer fully addresses the question by providing the common symptoms of appendicitis and stating that it cannot be cured via medicine. It also correctly identifies the surgical procedure to treat appendicitis, which is an open or laparoscopic appendectomy.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [37]:
groundedness, relevance = generate_ground_relevance_response(query_3)
print("GROUNDEDNESS:", groundedness)
print("RELEVANCE:", relevance)


GROUNDEDNESS:  I would rate the answer as 4 out of 5 for groundedness. The answer provides a comprehensive list of effective treatments for sudden patchy hair loss, which is supported by the context. Additionally, the answer correctly identifies the possible causes of sudden patchy hair loss as autoimmune disorders affecting genetically susceptible people exposed to unclear environmental triggers. However, the answer could be improved by providing more specific information about the environmental triggers that may cause alopecia areata.
RELEVANCE:  RELEVANCE: 4

The answer provides a comprehensive list of effective treatments for sudden patchy hair loss, which is relevant to the question. It also mentions possible causes, including autoimmune disorders and environmental triggers, which is also relevant. However, the answer could be improved by providing more information on each treatment option and its effectiveness, as well as more details on the possible causes and their mechanisms.


### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
groundedness, relevance = generate_ground_relevance_response(query_4)
print("GROUNDEDNESS:", groundedness)
print("RELEVANCE:", relevance)


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [39]:
groundedness, relevance = generate_ground_relevance_response(query_5)
print("GROUNDEDNESS:", groundedness)
print("RELEVANCE:", relevance)


GROUNDEDNESS:  GROUNDEDNESS: 1

The answer is not grounded in the context provided. The context discusses the care of a stump and prosthesis, as well as possible problems that may arise during rehabilitation, but it does not provide any information about what to do in the case of a fractured leg.
RELEVANCE:  RELEVANCE: 1 (not relevant)

The answer is not relevant to the question as it does not provide any information or guidance on the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, or their care and recovery.


## Actionable Insights and Business Recommendations

## Actionable Insights & Business Recommendations

**1. RAG clearly beats the plain model.**
Across the five questions, the answers changed a lot between the three approaches:
- **Plain LLM (no context):** general and sometimes vague, with a real risk of "hallucination" (made-up facts), because the model answers only from memory.
- **Prompt engineering:** clearer and better structured, but the facts still come from the model's memory, not from the manual.
- **RAG:** answers are tied to the retrieved pages of the Merck Manual, so they are more specific, more trustworthy, and traceable to a source. For a medical use-case, this **traceability** is the single most important benefit.

**2. The evaluation confirms quality, but also shows limits.**
Using the model as a judge (*LLM-as-a-judge*) on **groundedness** and **relevance** is a fast and cheap way to monitor quality. But the judge is the same Mistral model rating itself, so the scores are **indicative, not a replacement for expert review**. Recommendation: keep the automatic scores as a first filter, and send any low-scoring answer to a clinician for validation.

**3. Business value.**
- Faster access to reliable knowledge for healthcare professionals, which reduces information overload.
- The knowledge base is updated by simply **replacing the PDF** - no expensive model re-training.
- The system runs **locally**, so no patient data is sent to an external API (better privacy and compliance).

**4. Limitations to state honestly.**
- The system can only answer what is in the manual; missing topics correctly return "I don't know".
- Retrieval quality depends on the **chunk size** and on **k** (number of chunks): too few -> missing context; too many -> noise.

**5. Recommended next steps.**
- Add more trusted medical sources and tag them, so answers can show **which source** they came from.
- Tune chunk size / overlap, and compare **k = 3 vs k = 5** on the harder questions.
- Add a **human-in-the-loop** review for low groundedness / relevance scores before any clinical use.
- Log questions and scores to monitor quality over time.
- Run a formal **clinical safety and governance review** before deployment.


## Appendix - Setup Troubleshooting (Embedding / numpy fix)

This is a short and honest write-up of the real problem met during setup and how it was solved.

**The symptom.** Running the embedding cell raised:
`ImportError: cannot import name '_center' from 'numpy._core.umath'`,
followed by a second, **misleading** message: *"Could not import sentence_transformers ... pip install sentence-transformers"*. That second message looked like a missing package, but the package was installed - the real problem was **numpy**.

**The real cause.** `numpy` was in a **half-replaced (corrupted) state**: the Python files were the new version (2.3.3) but the compiled binary (`umath`) was from an older numpy that does **not** contain `_center`. Because the two did not match, every library built on numpy failed to import in this chain: `sentence-transformers -> scikit-learn -> scipy -> numpy`. This corruption came from the **base template's GPU install cell**, which used `--force-reinstall` with a CMAKE rebuild and disturbed numpy.

**The fix (realistic steps).**
1. **Remove the destructive GPU install cell** (`CMAKE_ARGS ... --force-reinstall`) and use a simple `!pip install -q llama-cpp-python` instead.
2. **Cleanly reinstall numpy** (the double uninstall removes any stacked copy):
   `!pip uninstall -y numpy`  (run twice)  then  `!pip install --no-cache-dir numpy==2.3.3`
3. **Restart the runtime** (Runtime -> Restart). Essential: the broken numpy stays in memory until a restart.
4. **Verify:** `from numpy._core.umath import _center` should no longer fail.
5. **Run the notebook top to bottom.** A `name 'llm' is not defined` error simply means a cell above (the model-loading cell) was not run yet.
6. Use the **modern imports** `langchain_huggingface.HuggingFaceEmbeddings` and `langchain_chroma.Chroma` to remove the deprecation warnings.

**Lesson learned.** Most of the "embedding" errors were **not about embeddings at all** - t**hey were a numpy / runtime-ordering problem**. Keeping numpy at one consistent version (2.3.3), never force-reinstalling it, a**nd always restarting after the install cells made the whole pipeline reproducible.**

<font size=6 color='blue'>Power Ahead</font>
___